In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv("twitter_training.csv", header=None)
val_df = pd.read_csv("twitter_validation.csv", header=None)

In [3]:
train_df.columns = ['id', 'entity', 'sentiment', 'text']
val_df.columns = ['id', 'entity', 'sentiment', 'text']

In [4]:
train_df.head()

,id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [5]:
val_df.head()

,id,entity,sentiment,text
0,3364,Facebook,Irrelevant,I mentioned on Facebook that I was struggling ...
1,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
3,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
4,4433,Google,Neutral,Now the President is slapping Americans in the...


In [6]:
print(train_df.columns)

Index(['id', 'entity', 'sentiment', 'text'], dtype='object')


In [7]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z\s!?]', '', text)  # keep emotion markers
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['text'] = train_df['text'].astype(str).apply(clean_text)
val_df['text'] = val_df['text'].astype(str).apply(clean_text)

In [8]:
label_map = {
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2
}

train_df['label'] = train_df['sentiment'].map(label_map)
val_df['label'] = val_df['sentiment'].map(label_map)

In [9]:
train_df = train_df[train_df['sentiment'].isin(['Negative', 'Neutral','Positive'])]
val_df = val_df[val_df['sentiment'].isin(['Negative', 'Neutral','Positive'])]

In [10]:
print(train_df['label'].isna().sum())  # should be 0
train_df = train_df.dropna(subset=['label'])
val_df = val_df.dropna(subset=['label'])


0


In [11]:
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)

In [12]:
train_df.head()

,id,entity,sentiment,text,label
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,2
1,2401,Borderlands,Positive,i am coming to the borders and i will kill you...,2
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you all,2
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,2
4,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,2


In [13]:
val_df.head()

,id,entity,sentiment,text,label
1,352,Amazon,Neutral,bbc news amazon boss jeff bezos rejects claims...,1
2,8312,Microsoft,Negative,why do i pay for word when it functions so poo...,0
3,4371,CS-GO,Negative,csgo matchmaking is so full of closet hacking ...,0
4,4433,Google,Neutral,now the president is slapping americans in the...,1
5,6273,FIFA,Negative,hi ive had madeleine mccann in my cellar for t...,0


In [14]:
print(train_df['sentiment'].unique())

['Positive' 'Neutral' 'Negative']


In [15]:
print(train_df['label'].value_counts())

label
0    22542
2    20832
1    18318
Name: count, dtype: int64


In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [17]:
lengths = train_df['text'].apply(lambda x: len(x.split()))
print(lengths.describe())

count    61692.000000
mean        17.481197
std         13.472106
min          0.000000
25%          7.000000
50%         14.000000
75%         25.000000
max        166.000000
Name: text, dtype: float64


In [18]:
import numpy as np
p90 = np.percentile(train_df['text'].apply(lambda x: len(x.split())), 90)
print(p90)  # likely around 35-40 based on your std

38.0


In [19]:
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])

max_len = 40

X_train = pad_sequences(X_train, maxlen=max_len, padding='post')
X_val = pad_sequences(X_val, maxlen=max_len, padding='post')

y_train = train_df['label']
y_val = val_df['label']

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D

model = Sequential([
    Embedding(20000, 128, input_length=max_len),

    Bidirectional(LSTM(192, return_sequences=True)),
    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

C:\Users\amanp\OneDrive\Desktop\dl proj\dlvenv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.build(input_shape=(None, max_len))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 40, 128)             │       2,560,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 40, 384)             │         493,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 384)                 │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │          24,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 3)                   │              99 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,079,875 (11.75 MB)

 Trainable params: 3,079,875 (11.75 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights_array))
print(class_weights)

{0: np.float64(0.9122526838789815), 1: np.float64(1.1226116388251992), 2: np.float64(0.9871351766513057)}


In [23]:

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/5
964/964 ━━━━━━━━━━━━━━━━━━━━ 161s 159ms/step - accuracy: 0.6831 - loss: 0.7365 - val_accuracy: 0.9215 - val_loss: 0.2295 - learning_rate: 0.0010
Epoch 2/5
964/964 ━━━━━━━━━━━━━━━━━━━━ 151s 157ms/step - accuracy: 0.8704 - loss: 0.3525 - val_accuracy: 0.9674 - val_loss: 0.1088 - learning_rate: 0.0010
Epoch 3/5
964/964 ━━━━━━━━━━━━━━━━━━━━ 150s 156ms/step - accuracy: 0.9151 - loss: 0.2244 - val_accuracy: 0.9710 - val_loss: 0.1120 - learning_rate: 0.0010
Epoch 4/5
964/964 ━━━━━━━━━━━━━━━━━━━━ 153s 158ms/step - accuracy: 0.9328 - loss: 0.1704 - val_accuracy: 0.9686 - val_loss: 0.1156 - learning_rate: 0.0010
Epoch 5/5
964/964 ━━━━━━━━━━━━━━━━━━━━ 150s 155ms/step - accuracy: 0.9509 - loss: 0.1177 - val_accuracy: 0.9771 - val_loss: 0.1260 - learning_rate: 5.0000e-04


In [24]:
from sklearn.metrics import classification_report

y_pred = np.argmax(model.predict(X_val), axis=1)
print(classification_report(y_val, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step
              precision    recall  f1-score   support

    Negative       0.97      0.98      0.98       266
     Neutral       0.97      0.98      0.98       285
    Positive       0.99      0.97      0.98       277

    accuracy                           0.98       828
   macro avg       0.98      0.98      0.98       828
weighted avg       0.98      0.98      0.98       828



In [25]:
import pickle
# Save model
model.save("sentiment_model.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [26]:
import numpy as np

def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)

    pred = model.predict(padded)[0]
    label = np.argmax(pred)

    mapping = {0: "Negative", 1: "Neutral", 2: "Positive"}

    print("Probabilities:", pred)

    return mapping[label]

In [27]:
print(predict_sentiment("I hate this so much"))
print(predict_sentiment("This is okay"))
print(predict_sentiment("Worst experience of my life"))
print(predict_sentiment("I absolutely love this"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
Probabilities: [0.30848348 0.47713315 0.2143833 ]
Neutral
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
Probabilities: [0.7930714  0.17776033 0.02916823]
Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Probabilities: [0.1513807  0.77231145 0.07630786]
Neutral
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Probabilities: [0.1532981  0.11327371 0.7334282 ]
Positive


In [28]:
print(train_df['sentiment'].value_counts())


sentiment
Negative    22542
Positive    20832
Neutral     18318
Name: count, dtype: int64


In [29]:
print(train_df[['sentiment', 'label']].drop_duplicates().sort_values('label'))

   sentiment  label
24  Negative      0
12   Neutral      1
0   Positive      2


In [30]:
print(len(history.history['val_accuracy']))   # epochs ran
print(max(history.history['val_accuracy']))   # best accuracy
print(history.history['val_accuracy'])        # all epochs

5
0.977053165435791
[0.9214975833892822, 0.967391312122345, 0.9710144996643066, 0.9685990214347839, 0.977053165435791]


In [31]:
print(predict_sentiment("Borderlands is amazing I love this game"))
print(predict_sentiment("This phone is terrible never buying again"))
print(predict_sentiment("The update was okay nothing special"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
Probabilities: [0.00783422 0.01232471 0.9798411 ]
Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Probabilities: [0.80119675 0.07778542 0.12101775]
Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Probabilities: [0.6076021  0.19291721 0.19948067]
Negative


In [32]:
from sklearn.metrics import classification_report
y_pred = np.argmax(model.predict(X_val), axis=1)
print(classification_report(y_val, y_pred, 
      target_names=['Negative', 'Neutral', 'Positive']))

26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step
              precision    recall  f1-score   support

    Negative       0.97      0.98      0.98       266
     Neutral       0.97      0.98      0.98       285
    Positive       0.99      0.97      0.98       277

    accuracy                           0.98       828
   macro avg       0.98      0.98      0.98       828
weighted avg       0.98      0.98      0.98       828



In [57]:
print(predict_sentiment("Apple's new iPhone is absolutely amazing"))
print(predict_sentiment("This game keeps crashing it's so frustrating"))
print(predict_sentiment("The service was bad"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step
Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Negative


In [58]:
test_sentences = [
    ("Apple's new iPhone is absolutely amazing", "Positive"),
    ("This game keeps crashing so frustrating",  "Negative"),
    ("The service was okay nothing special",      "Neutral"),
    ("I love this product best purchase ever",    "Positive"),
    ("Worst update they have ever released",      "Negative"),
]

print(f"{'Text':<45} {'Expected':<12} {'Predicted':<12} {'Match'}")
print("-" * 80)

for text, expected in test_sentences:
    predicted = predict_sentiment(text)
    match = "ok" if predicted == expected else "X"
    print(f"{text:<45} {expected:<12} {predicted:<12} {match}")

Text                                          Expected     Predicted    Match
--------------------------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Apple's new iPhone is absolutely amazing      Positive     Positive     ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
This game keeps crashing so frustrating       Negative     Negative     ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
The service was okay nothing special          Neutral      Negative     X
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
I love this product best purchase ever        Positive     Positive     ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Worst update they have ever released          Negative     Negative     ok


In [59]:
test_sentences = [
    ("Samsung galaxy is the best phone ever",     "Positive"),
    ("Netflix keeps buffering so annoying",        "Negative"),
    ("The movie was just average nothing great",   "Neutral"),
    ("Amazon delivery was super fast loved it",    "Positive"),
    ("Twitter is down again this is ridiculous",   "Negative"),
    ("The food was neither good nor bad",          "Neutral"),
]

print(f"{'Text':<45} {'Expected':<12} {'Predicted':<12} {'Match'}")
print("-" * 80)

for text, expected in test_sentences:
    predicted = predict_sentiment(text)
    match = "ok" if predicted == expected else "X"
    print(f"{text:<45} {expected:<12} {predicted:<12} {match}")

Text                                          Expected     Predicted    Match
--------------------------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Samsung galaxy is the best phone ever         Positive     Positive     ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Netflix keeps buffering so annoying           Negative     Negative     ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
The movie was just average nothing great      Neutral      Neutral      ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Amazon delivery was super fast loved it       Positive     Positive     ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Twitter is down again this is ridiculous      Negative     Negative     ok
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
The food was neither good nor bad             Neutral      Negative     X


In [36]:
casual_sentences = [
    "he is not a good boy",
    "she is so kind and helpful",
    "this is the worst day of my life",
    "i am feeling okay today",
    "i am so happy right now",
    "this is completely useless",
    "it was an average experience",
    "i cant believe how bad this is",
]

for text in casual_sentences:
    predicted = predict_sentiment(text)
    print(f"{text:<45} → {predicted}")
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Probabilities: [0.36541402 0.1431928  0.49139315]
he is not a good boy                          → Positive

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
Probabilities: [0.4770018  0.46107754 0.06192069]
she is so kind and helpful                    → Negative

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
Probabilities: [0.00910151 0.96226436 0.02863414]
this is the worst day of my life              → Neutral

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Probabilities: [0.41735214 0.5311724  0.05147548]
i am feeling okay today                       → Neutral

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Probabilities: [0.16849473 0.6324355  0.19906977]
i am so happy right now                       → Neutral

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
Probabilities: [0.31884927 0.39336663 0.28778404]
this is completely useless                    → Neutral

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
Probabilities: [0.01500375 0.9657012  0.019295  ]
it was an average experience         

In [56]:
# !pip install transformers torch

In [38]:
from transformers import pipeline

sentiment = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

def predict_general(text):
    result = sentiment(text)[0]

    label_map = {
        'LABEL_0': 'Negative',
        'LABEL_1': 'Neutral',
        'LABEL_2': 'Positive'
    }

    label = label_map[result['label']]
    score = result['score'] * 100

    print(f"Text      : {text}")
    print(f"Prediction: {label} ({score:.2f}%)")
    print("-" * 40)

    return label

C:\Users\amanp\OneDrive\Desktop\dl proj\dlvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 3945.67it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\amanp\OneDrive\Desktop\dl proj\dlvenv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently

In [42]:
print(sentiment("I hate this so much"))
print(sentiment("This is the best day ever"))
print(sentiment("okay that was yeah"))

[{'label': 'negative', 'score': 0.8875116109848022}]
[{'label': 'positive', 'score': 0.9799087643623352}]
[{'label': 'positive', 'score': 0.6045534610748291}]


In [49]:
print(sentiment("he is a boy"))
print(sentiment("she was not so  kind and helpful"))
print(sentiment("this is the worst day of my life"))
print(sentiment("the staff was rude and unprofessional"))

[{'label': 'neutral', 'score': 0.7576023936271667}]
[{'label': 'negative', 'score': 0.8395519256591797}]
[{'label': 'negative', 'score': 0.9247579574584961}]
[{'label': 'negative', 'score': 0.8867096900939941}]


In [50]:
print(predict_sentiment("this is the worst day of my life"))  #lstm
print(sentiment("this is the worst day of my life"))          # Transformer

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Probabilities: [0.00910151 0.96226436 0.02863414]
Neutral
[{'label': 'negative', 'score': 0.9247579574584961}]


In [51]:
!pip install streamlit

     ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.2 MB 1.4 MB/s eta 0:00:07
     ---------------------------------------- 0.0/9.2 MB 1.4 MB/s eta 0:00:07
     ---------------------------------------- 0.0/9.2 MB 1.4 MB/s eta 0:00:07
     ---------------------------------------- 0.1/9.2 MB 297.7 kB/s eta 0:00:31
     ---------------------------------------- 0.1/9.2 MB 469.7 kB/s eta 0:00:20
     ---------------------------------------- 0.1/9.2 MB 469.7 kB/s eta 0:00:20
      --------------------------------------- 0.2/9.2 MB 468.3 kB/s eta 0:00:20
     - -------------------------------------- 0.2/9.2 MB 656.0 kB/s eta 0:00:14
     - -------------------------------------- 0.3/9.2 MB 710.0 kB/s eta 0:00:13
     - -------------------------------------- 0.3/9.2 MB 726.4 kB/s eta 0:00:13
     -- ------------------------------------- 0.5/9.2 MB 962.6 kB/s eta 0:00:10
     -- ------------------------------------- 0.6/9.2 M


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
model.save("lstm_model.keras")

In [54]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)